# FIT5196 Assessment 1 — Group030 Solution
Complete structured parsing, transformation, reconciliation and validation evidence. **Members must add names and student IDs before submission.**

## 0. Configuration and reproducibility
All paths are relative/configurable; the workflow performs no network I/O.

In [1]:
from pathlib import Path
import json, hashlib, xml.etree.ElementTree as ET
import pandas as pd
GROUP_ID='Group030'; INPUT_DIR=Path('raw_input'); OUTPUT_DIR=Path('outputs'); TEMPLATE_DIR=Path('templates'); DICTIONARY_PATH=Path('public_data_dictionary.csv')
OUTPUT_DIR.mkdir(exist_ok=True)
from Group030_solution import build_tables, validate
from Group030_text_functions import *
print({'group':GROUP_ID,'input':str(INPUT_DIR),'output':str(OUTPUT_DIR),'pandas':pd.__version__})

{'group': 'Group030', 'input': 'raw_input', 'output': 'outputs', 'pandas': '3.0.5'}


### 0.1 Package integrity
The archive, manifest and source filenames must all identify Group030; hashes verify that the allocated inputs were not altered.

In [2]:
manifest=json.load(open('A1_manifest.json'))
checks=[]
for entry in manifest['files']:
 p=Path(entry['path']); checks.append({'path':entry['path'],'exists':p.exists(),'bytes':p.stat().st_size if p.exists() else None,'sha256_match':p.exists() and hashlib.sha256(p.read_bytes()).hexdigest()==entry['sha256']})
pd.DataFrame(checks)

,path,exists,bytes,sha256_match
0,README.md,True,945,False
1,public_data_dictionary.csv,True,11151,True
2,raw_input/Group030_commerce.json,True,14284869,True
3,raw_input/Group030_operations.xml,True,17863595,True


## 1. Parse and profile the two sources
Both documents are parsed with structured parsers. Regex is never used to reconstruct JSON/XML.

### 1.1 JSON structure and grain
`customerProfiles` is one customer per element; `orders` is one order with nested header, repeated cart items and one delivery; `productReviews` is one review. Candidate keys are customerID, orderID, orderItemID, deliveryID and reviewID. JSON uses ISO dates/timestamps, Python booleans, numeric AUD values and empty strings for optional text.

In [3]:
with open(INPUT_DIR/f'{GROUP_ID}_commerce.json',encoding='utf-8') as f: js=json.load(f)
json_profile=pd.DataFrame([{'collection':k,'python_type':type(v).__name__,'records':len(v) if isinstance(v,list) else 1,'top_fields':' | '.join(v[0].keys()) if isinstance(v,list) and v else ' | '.join(v.keys())} for k,v in js.items()])
json_profile

,collection,python_type,records,top_fields
0,customerProfiles,list,500,accountStatus | acquisitionSource | ageBand | ...
1,exportMetadata,dict,1,groupAlias | period | sourceSystem
2,orders,list,2818,delivery | header | shoppingCart
3,productReviews,list,3946,customerID | deliveryExperience | helpfulVotes...


### 1.2 XML structure and grain
`OperationsExport/Orders/Order` repeats orders; each contains Header, repeated Shopping_Cart/Item and one Delivery. ProductCatalogue/Product and ProductReviews/Review repeat at product and review grain. XML uses day-first dates, Y/N booleans, AUD labels, comma thousands separators, percentage strings and empty elements.

In [4]:
root=ET.parse(INPUT_DIR/f'{GROUP_ID}_operations.xml').getroot()
xml_profile=pd.DataFrame([('Orders/Order','order',len(root.findall('./Orders/Order'))),('Shopping_Cart/Item','order item',len(root.findall('./Orders/Order/Shopping_Cart/Item'))),('Order/Delivery','delivery',len(root.findall('./Orders/Order/Delivery'))),('ProductCatalogue/Product','product',len(root.findall('./ProductCatalogue/Product'))),('ProductReviews/Review','review',len(root.findall('./ProductReviews/Review')))],columns=['path','grain','records'])
xml_profile

,path,grain,records
0,Orders/Order,order,2818
1,Shopping_Cart/Item,order item,8885
2,Order/Delivery,delivery,2818
3,ProductCatalogue/Product,product,1000
4,ProductReviews/Review,review,3946


### 1.3 Source comparison, overlap and assumptions
JSON uniquely supplies customers; XML uniquely supplies products. Orders, items, deliveries and reviews overlap partially across sources. Comparable fields are normalised first, then compared by primary key. Equal records collapse; any differing non-missing value is a validation conflict. No source-precedence rule is used. IDs retain case/leading zeros; missing prescribed strings become literal `NaN`.

In [5]:
tables,profile=build_tables(INPUT_DIR,DICTIONARY_PATH)
pd.DataFrame(profile['json'].items(),columns=['entity','JSON rows']).merge(pd.DataFrame(profile['xml'].items(),columns=['entity','XML rows']),on='entity',how='outer').fillna(0), profile['canonical']

(        entity  JSON rows  XML rows
 0    customers      500.0       0.0
 1   deliveries     2818.0    2818.0
 2  order_items     8823.0    8885.0
 3       orders     2818.0    2818.0
 4     products        0.0    1000.0
 5      reviews     3946.0    3946.0,
 {'orders': 5000,
  'order_items': 15723,
  'customers': 500,
  'deliveries': 5000,
  'products': 1000,
  'product_reviews': 7000})

## 2. Source-to-target mapping
All 111 target fields retain template MAP IDs. Paths are structural, derivations specify formulas and all multi-source rows document comparison/conflict behaviour.

In [6]:
mapping=pd.read_csv('Group030_source_to_target_mapping.csv',keep_default_na=False)
core=['source_format','transformation_or_derivation','overlap_or_conflict_rule','notebook_evidence']
summary=mapping.groupby('output_table').agg(required_rows=('mapping_id','size'),core_complete=('mapping_id',lambda x:0))
summary['core_complete']=[mapping[mapping.output_table.eq(t)][core].ne('').all(axis=1).sum() for t in summary.index]
summary

,required_rows,core_complete
output_table,,
customers,20,20
deliveries,20,20
order_items,6,6
orders,23,23
product_reviews,21,21
products,21,21


## 3. Text and regex functions
Processing order is: structured field retrieval → reference extraction → entity decode/NFC → tags → published markers/URLs/emoji → complete reference wrapper → promo wrapper → whitespace/lowercase → literal `NaN`. `review_body_clean` preserves multilingual letters; Latin analysis is derived separately.

In [7]:
cases=pd.read_csv(TEMPLATE_DIR/'A1_public_text_test_cases.csv',keep_default_na=False)
results=[]
for _,x in cases.iterrows():
 actual=str(globals()[x.function](x.input_value));results.append({'case_id':x.case_id,'function':x.function,'actual':actual,'expected':x.expected_output,'status':'PASS' if actual==x.expected_output else 'FAIL'})
public_results=pd.DataFrame(results);public_results

,case_id,function,actual,expected,status
0,TXT-01,clean_narrative_text,leave at reception,leave at reception,PASS
1,TXT-02,extract_promo_code,B3SAVE-24,B3SAVE-24,PASS
2,TXT-03,clean_narrative_text,café setup was easy,café setup was easy,PASS
3,TXT-04,extract_order_reference,HORD123456,HORD123456,PASS
4,TXT-05,extract_product_sku,SKU-ABC123,SKU-ABC123,PASS
5,TXT-06,extract_order_reference,NaN,NaN,PASS
6,TXT-07,build_latin_analysis,service était bon,service était bon,PASS
7,TXT-08,contains_non_latin_script,True,True,PASS
8,TXT-09,build_latin_analysis,NaN,NaN,PASS
9,TXT-10,clean_narrative_text,reliable for daily use,reliable for daily use,PASS


In [8]:
student_tests=[('missing',clean_narrative_text(None),'NaN'),('Latin diacritic',build_latin_analysis('déjà 東京'),'déjà'),('non-Latin flag',contains_non_latin_script('déjà 東京'),True),('embedded order',extract_order_reference('XHORD123456'),'NaN'),('extended SKU',extract_product_sku('SKU-ABC_extra'),'NaN'),('extended promo',extract_promo_code('B3SAVE-24_more'),'NaN')]
pd.DataFrame(student_tests,columns=['case','actual','expected']).assign(status=lambda x:x.actual.eq(x.expected).map({True:'PASS',False:'FAIL'}))

,case,actual,expected,status
0,missing,NaN,NaN,PASS
1,Latin diacritic,déjà,déjà,PASS
2,non-Latin flag,True,True,PASS
3,embedded order,NaN,NaN,PASS
4,extended SKU,NaN,NaN,PASS
5,extended promo,NaN,NaN,PASS


## 4. Build the six standardised relational tables
The maintained implementation is `Group030_solution.py`; the following cells expose each table's grain, field order, row flow and material derivations.

### 4.1 Orders
One canonical order per `order_id`. Timestamps, percentages, booleans, coordinates and narrative are normalised. `order_price` is recomputed from rounded canonical lines; GST is included `order_price/11`; total applies discount then delivery.

In [9]:
tables['orders'].head(3), {'rows':len(tables['orders']),'unique_keys':tables['orders'].order_id.nunique(),'columns':tables['orders'].columns.tolist()}

(     order_id source_system_record_id customer_id      order_timestamp  \
 0  HORD000001        SRC-030-H-000001    CUS00225  2018-09-07 10:05:00   
 1  HORD000002        SRC-030-H-000002    CUS00099  2018-02-23 21:51:00   
 2  HORD000003        SRC-030-H-000003    CUS00238  2018-08-05 13:53:00   
 
   sales_channel payment_method currency nearest_warehouse order_status  \
 0         Store         PayPal      AUD          Thompson    Completed   
 1           Web         PayPal      AUD            Bakers    Completed   
 2           Web           Card      AUD         Nickolson    Completed   
 
    order_price  ...  tax_amount order_total  season  expedited_delivery  \
 0      6916.92  ...      628.81     6931.21  Spring               False   
 1      1867.88  ...      169.81     1878.88  Summer               False   
 2      7032.93  ...      639.36     7061.68  Winter                True   
 
    customer_lat customer_long  device_type  referral_source  \
 0    -37.882177    144.88

### 4.2 Order items
One row per `order_item_id`; nested arrays are flattened without joining reviews. `line_revenue = round(quantity × unit_price, 2)`.

In [10]:
tables['order_items'].head(3),tables['order_items'].agg({'quantity':['min','max'],'unit_price':['min','max'],'line_revenue':['min','max']})

(  order_item_id    order_id product_id  quantity  unit_price  line_revenue
 0   HITM0000001  HORD000001    PRD0054         2     2814.02       5628.04
 1   HITM0000002  HORD000001    PRD0466         1      520.19        520.19
 2   HITM0000003  HORD000001    PRD0628         1       62.19         62.19,
      quantity  unit_price  line_revenue
 min         1       15.23         15.23
 max         3     4182.34      12547.02)

### 4.3 Customers
One row per JSON customer. Postcodes remain strings so leading zeros would survive; consent is boolean and signup date is ISO.

In [11]:
tables['customers'].head(3),tables['customers'][['loyalty_tier','customer_segment','preferred_language']].describe()

(  customer_id signup_date loyalty_tier customer_segment age_band  \
 0    CUS00001  2016-03-17       Silver   Small Business    25-34   
 1    CUS00002  2017-12-10       Bronze       Mainstream    45-54   
 2    CUS00003  2014-09-05         Gold       Mainstream    35-44   
 
   preferred_channel home_suburb  prior_12m_orders  \
 0             Store    Hawthorn                 5   
 1             Store    Hawthorn                14   
 2            Mobile    St Kilda                 2   
 
    lifetime_value_before_period  marketing_consent home_postcode home_state  \
 0                       1876.75              False          3122        VIC   
 1                       5191.82               True          3122        VIC   
 2                       2135.34               True          3182        VIC   
 
   home_country preferred_language acquisition_source account_status  \
 0    Australia                 nl        Paid Search         Active   
 1    Australia                 de    

### 4.4 Deliveries
One completed delivery per `delivery_id`; XML/JSON dates and boolean alternatives are reconciled. Narrative is cleaned through the same bounded function.

In [12]:
tables['deliveries'].head(3),tables['deliveries'].groupby(['carrier','service_level']).size().rename('rows')

(  delivery_id    order_id dispatch_date promised_date delivered_date  \
 0  HDEL000001  HORD000001    2018-09-08    2018-09-13     2018-09-13   
 1  HDEL000002  HORD000002    2018-02-24    2018-03-01     2018-02-27   
 2  HDEL000003  HORD000003    2018-08-07    2018-08-12     2018-08-10   
 
           carrier service_level delivery_status  delay_days  on_time_in_full  \
 0         AusPost      Standard       Delivered           0             True   
 1  Direct Freight      Standard       Delivered           0             True   
 2         AusPost       Express       Delivered           0             True   
 
    fulfilment_hours  delivery_cost delay_reason  promised_days  \
 0                30          10.20         none              5   
 1                19           7.67         none              5   
 2                48          21.04         none              5   
 
    tracking_event_count delivery_window  shipping_distance_km  \
 0                     9       Afternoon    

### 4.5 Products
One XML catalogue row per product. AUD cost/price, dates, flags and product description are standardised.

In [13]:
tables['products'].head(3),tables['products'].groupby('category').agg(products=('product_id','size'),mean_price=('unit_price','mean'))

(  product_id      product_name    category   brand  unit_price  unit_cost  \
 0    PRD0001  Candle Bloom 100      Laptop  Candle     2393.92    1475.97   
 1    PRD0002     Vela Halo 101  Smartphone    Vela     1391.38     905.35   
 2    PRD0003  Candle Quest 102      Tablet  Candle      316.95     160.31   
 
    launch_year  warranty_months  weight_kg   product_sku  ... model_family  \
 0         2014               24      1.196  SKU-CAN00001  ...          Arc   
 1         2012               24      0.261  SKU-VEL00002  ...        Atlas   
 2         2017               24      0.610  SKU-CAN00003  ...        Bloom   
 
      colour supplier_id supplier_country launch_date  tax_category  \
 0    Silver      SUP001          Vietnam  2014-01-25  GST_STANDARD   
 1     Green      SUP002         Malaysia  2012-08-14  GST_STANDARD   
 2  Graphite      SUP003            Korea  2017-10-07  GST_STANDARD   
 
       package_type recyclable_packaging  active_flag  \
 0     Recycled box      

### 4.6 Product reviews
One canonical review per `review_id`; structured attributes are reconciled and raw text yields multilingual clean text, Latin analysis, references, flags and measures.

In [14]:
tables['product_reviews'].head(3),tables['product_reviews'].groupby(['language_code','contains_non_latin_script']).size().sort_values(ascending=False).head(12)

(    review_id    order_id order_item_id product_id customer_id  \
 0  HREV000001  HORD000001   HITM0000001    PRD0054    CUS00225   
 1  HREV000002  HORD000001   HITM0000003    PRD0628    CUS00225   
 2  HREV000003  HORD000001   HITM0000004    PRD0620    CUS00225   
 
       review_timestamp language_code  rating  \
 0  2018-09-15 11:01:00            en       5   
 1  2018-10-28 12:02:00            en       5   
 2  2018-10-13 13:03:00            en       3   
 
                           review_title  \
 0  a pleasant shared-screen experience   
 1     helpful summaries after exercise   
 2    protective case that travels well   
 
                                    review_body_clean  ... verified_purchase  \
 0  vela spark 153 has become the screen we use mo...  ...              True   
 1  i have used vela echo 727 after walks, short r...  ...              True   
 2  ⭐ vela atlas 719 has carried my tablet and cha...  ...              True   
 
    helpful_votes  review_length_cha

## 5. Reconciliation and relationships
Raw-to-canonical counts demonstrate partial overlap. Reconciliation is key-driven and deterministic; zero conflicts means overlapping values agreed after normalisation.

In [15]:
flow=[]
for table,rawname in [('orders','orders'),('order_items','order_items'),('deliveries','deliveries'),('product_reviews','reviews')]:
 raw=profile['json'][rawname]+profile['xml'][rawname];flow.append({'table':table,'JSON':profile['json'][rawname],'XML':profile['xml'][rawname],'raw_total':raw,'canonical':len(tables[table]),'overlap_removed':raw-len(tables[table])})
pd.DataFrame(flow),{'normalised_conflicts':len(profile['conflicts'])}

(             table  JSON   XML  raw_total  canonical  overlap_removed
 0           orders  2818  2818       5636       5000              636
 1      order_items  8823  8885      17708      15723             1985
 2       deliveries  2818  2818       5636       5000              636
 3  product_reviews  3946  3946       7892       7000              892,
 {'normalised_conflicts': 0})

## 6. Validation register
Each executable check reports a stable ID, observed result, PASS/FAIL and resolution. Coverage includes schema/order, missing representation, PK/FK, row flow, conflict detection, arithmetic, ranges, temporal order, references and multilingual behaviour.

In [16]:
dictionary=pd.read_csv(DICTIONARY_PATH);validation_register=validate(tables,dictionary,profile)
validation_register

,validation_id,status,observed_result,resolution_or_interpretation
0,VAL-SCHEMA-01,PASS,"['customers', 'deliveries', 'order_items', 'or...",None required
1,VAL-SCHEMA-02,PASS,"orders: 5000 rows, 23 ordered columns",None required
2,VAL-PK-01,PASS,"orders.order_id: missing=0, duplicates=0",None required
3,VAL-MISS-01,PASS,orders: empty required values=0,None required
4,VAL-SCHEMA-03,PASS,"order_items: 15723 rows, 6 ordered columns",None required
5,VAL-PK-02,PASS,"order_items.order_item_id: missing=0, duplicat...",None required
6,VAL-MISS-02,PASS,order_items: empty required values=0,None required
7,VAL-SCHEMA-04,PASS,"customers: 500 rows, 20 ordered columns",None required
8,VAL-PK-03,PASS,"customers.customer_id: missing=0, duplicates=0",None required
9,VAL-MISS-03,PASS,customers: empty required values=0,None required


In [17]:
validation_register.groupby('status').size(), validation_register[validation_register.status.ne('PASS')]

(status
 PASS    45
 dtype: int64,
 Empty DataFrame
 Columns: [validation_id, status, observed_result, resolution_or_interpretation]
 Index: [])

## 7. Deterministic export
Only dictionary fields, in published order, enter the six submitted CSVs. Reading with `keep_default_na=False` confirms literal `NaN` is retained.

In [18]:
for name,df in tables.items():df.to_csv(OUTPUT_DIR/f'{GROUP_ID}_{name}_standardised.csv',index=False,na_rep='NaN')
validation_register.to_csv(OUTPUT_DIR/f'{GROUP_ID}_validation_register.csv',index=False)
roundtrip={n:list(pd.read_csv(OUTPUT_DIR/f'{GROUP_ID}_{n}_standardised.csv',keep_default_na=False).columns)==list(tables[n].columns) for n in tables}
roundtrip

{'orders': True,
 'order_items': True,
 'customers': True,
 'deliveries': True,
 'products': True,
 'product_reviews': True}

## 8. Final reproducibility record
Fresh offline execution recreates six CSVs and all validation evidence without manual edits, certified counts, live services or absolute paths.

In [19]:
assert public_results.status.eq('PASS').all();assert validation_register.status.eq('PASS').all();assert all(roundtrip.values());print('FINAL STATUS: PASS — public tests, validation register and round-trip schemas')

FINAL STATUS: PASS — public tests, validation register and round-trip schemas
